### 15/03/25  

#### 权重衰减与正则化

将权重的范数作为惩罚项加到最小化损失的问题中。
使原来的训练目标*最小化训练标签上的预测损失*，
调整为*最小化预测损失和惩罚项之和*。
现在，如果我们的权重向量增长的太大，
我们的学习算法可能会更集中于最小化权重范数$\| \mathbf{w} \|^2$。
这正是我们想要的。
让我们回顾一下线性回归的例子。
我们的损失由下式给出：

$$L(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^n \frac{1}{2}\left(\mathbf{w}^\top \mathbf{x}^{(i)} + b - y^{(i)}\right)^2.$$

我们必须以某种方式在损失函数中添加$\| \mathbf{w} \|^2$，
但是模型应该如何平衡这个新的额外惩罚的损失？
实际上，我们通过*正则化常数*$\lambda$来描述这种权衡，
这是一个非负超参数，我们使用验证数据拟合：

$$L(\mathbf{w}, b) + \frac{\lambda}{2} \|\mathbf{w}\|^2,$$

此外，为什么我们首先使用$L_2$范数，而不是$L_1$范数。
事实上，这个选择在整个统计领域中都是有效的和受欢迎的。
$L_2$正则化线性模型构成经典的*岭回归*（ridge regression）算法，
$L_1$正则化线性回归是统计学中类似的基本模型，
通常被称为*套索回归*（lasso regression）。
使用$L_2$范数的一个原因是它对权重向量的大分量施加了巨大的惩罚。
这使得我们的学习算法偏向于在大量特征上均匀分布权重的模型。
在实践中，这可能使它们对单个变量中的观测误差更为稳定。
相比之下，$L_1$惩罚会导致模型将权重集中在一小部分特征上，
而将其他权重清除为零。
这称为*特征选择*（feature selection），这可能是其他场景下需要的。

使用与 :eqref:`eq_linreg_batch_update`中的相同符号，
$L_2$正则化回归的小批量随机梯度下降更新如下式：

$$
\begin{aligned}
\mathbf{w} & \leftarrow \left(1- \eta\lambda \right) \mathbf{w} - \frac{\eta}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \mathbf{x}^{(i)} \left(\mathbf{w}^\top \mathbf{x}^{(i)} + b - y^{(i)}\right).
\end{aligned}
$$

#### Dropout  
当面对更多的特征而样本不足时，线性模型往往会过拟合。
相反，当给出更多样本而不是特征，通常线性模型不会过拟合。
不幸的是，线性模型泛化的可靠性是有代价的。
简单地说，线性模型没有考虑到特征之间的交互作用。
对于每个特征，线性模型必须指定正的或负的权重，而忽略其他特征。

泛化性和灵活性之间的这种基本权衡被描述为*偏差-方差权衡*（bias-variance tradeoff）。
线性模型有很高的偏差：它们只能表示一小类函数。
然而，这些模型的方差很低：它们在不同的随机数据样本上可以得出相似的结果。

### 模型参数访问

In [1]:
import torch
import torch.nn as nn 
import torch.functional as F 

net = nn.Sequential(nn.Linear(8, 4), nn.ReLU(), nn.Linear(4, 1))

X = torch.rand(2, 8)
net(X)


tensor([[0.2855],
        [0.2517]], grad_fn=<AddmmBackward0>)

In [2]:
print(net)

Sequential(
  (0): Linear(in_features=8, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)


In [6]:
# 访问参数

# print([(name, para.shape) for name, para in net[0].named_parameters()])
# print([(name, para.shape) for name, para in net.named_parameters()])
print([(name, para) for name, para in net.named_parameters()])

[('0.weight', Parameter containing:
tensor([[-0.0208, -0.1163, -0.2095, -0.0585, -0.3257, -0.1682,  0.0197,  0.0652],
        [-0.1059, -0.2417, -0.3351, -0.2140, -0.1467, -0.1576,  0.3168,  0.2859],
        [ 0.3533,  0.2828,  0.2630,  0.1767,  0.1919,  0.1431,  0.0136,  0.3117],
        [-0.3369, -0.2091,  0.3471,  0.2379,  0.3169, -0.1736,  0.0992, -0.2732]],
       requires_grad=True)), ('0.bias', Parameter containing:
tensor([-0.3014, -0.2168, -0.1563, -0.3483], requires_grad=True)), ('2.weight', Parameter containing:
tensor([[-0.3334,  0.0434, -0.2590,  0.2923]], requires_grad=True)), ('2.bias', Parameter containing:
tensor([0.4717], requires_grad=True))]


In [7]:
net.state_dict()['0.weight']

tensor([[-0.0208, -0.1163, -0.2095, -0.0585, -0.3257, -0.1682,  0.0197,  0.0652],
        [-0.1059, -0.2417, -0.3351, -0.2140, -0.1467, -0.1576,  0.3168,  0.2859],
        [ 0.3533,  0.2828,  0.2630,  0.1767,  0.1919,  0.1431,  0.0136,  0.3117],
        [-0.3369, -0.2091,  0.3471,  0.2379,  0.3169, -0.1736,  0.0992, -0.2732]])

### 参数初始化

In [10]:
def init_normal(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 1, 0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight, net[0].bias

(Parameter containing:
 tensor([[0.9995, 1.0262, 0.9944, 1.0099, 0.9976, 0.9962, 0.9852, 0.9976],
         [1.0040, 0.9885, 0.9999, 0.9979, 1.0125, 0.9893, 1.0122, 1.0007],
         [0.9981, 1.0015, 1.0202, 1.0150, 0.9929, 0.9851, 0.9857, 1.0025],
         [1.0024, 0.9840, 0.9938, 1.0014, 1.0033, 0.9769, 1.0009, 1.0043]],
        requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0.], requires_grad=True))

In [8]:
def init_normal(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=1, std=0.01)
        nn.init.zeros_(m.bias)  
net.apply(init_normal)        
net[0].weight, net[0].bias         

(Parameter containing:
 tensor([[0.9973, 0.9995, 1.0072, 1.0055, 1.0023, 1.0016, 1.0046, 0.9818],
         [1.0052, 0.9911, 0.9900, 0.9824, 1.0053, 0.9972, 0.9976, 1.0024],
         [0.9982, 0.9920, 1.0056, 0.9996, 1.0107, 1.0039, 0.9952, 1.0035],
         [0.9954, 1.0108, 0.9995, 1.0007, 0.9748, 1.0010, 1.0124, 0.9959]],
        requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0.], requires_grad=True))